# Massive Historical Underlying Access Probe — 2021 and 2022

This notebook answers a narrow question before any additional dissertation data are downloaded:

> **Does the current Massive account return one-minute SPY, SPX and VIX data for 2022 and/or 2021?**

It deliberately **does not write to the dissertation DuckDB database** and does not alter the existing raw-data folders.

A 2023 date is tested as a control because the current project expects 2023 access. The notebook then probes several ordinary trading dates in 2022 and 2021.

## Interpretation

For each ticker/year:

- `AVAILABLE` — at least one probe date returned aggregate rows;
- `NO_ROWS` — requests completed but no aggregate rows were returned;
- `ERROR_OR_ENTITLEMENT` — the API returned an exception/error;
- `MIXED` — different probe dates produced different outcomes.

Because data entitlements can differ between stocks and indices, SPY, SPX and VIX are reported separately.

If 2022 or 2021 works, we can then extend the main robustness pipeline further back without changing its methodology.

## 1. Environment and Massive client

In [ ]:
from pathlib import Path
import os
import sys
import json

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
ENV_PATH = PROJECT_ROOT / "main.env"
SRC_ROOT = PROJECT_ROOT / "src"

if not ENV_PATH.exists():
    raise FileNotFoundError(f"Missing environment file: {ENV_PATH}")

if not SRC_ROOT.exists():
    raise FileNotFoundError(
        "Missing src directory. This probe uses the same src/massive_database.py "
        "as the dissertation retrieval workflow."
    )

load_dotenv(ENV_PATH, override=True)
API_KEY = os.getenv("MASSIVE_API_KEY")

if not API_KEY:
    raise RuntimeError("MASSIVE_API_KEY is missing from main.env/environment.")

sys.path.insert(0, str(SRC_ROOT))

from massive_database import MassiveREST, normalize_aggregates

client = MassiveREST(API_KEY)

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "underlying_history_probe"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Probe output:", OUTPUT_ROOT)
print("API key loaded:", bool(API_KEY))

## 2. Probe dates

In [ ]:
# Multiple ordinary weekdays are used so that one unusual/missing session
# does not determine the conclusion for an entire year.
PROBE_DATES = {
    2023: [
        "2023-02-15",
        "2023-05-15",
        "2023-08-15",
        "2023-11-15",
    ],
    2022: [
        "2022-02-15",
        "2022-05-16",
        "2022-08-15",
        "2022-11-15",
    ],
    2021: [
        "2021-02-16",
        "2021-05-17",
        "2021-08-16",
        "2021-11-15",
    ],
}

TICKERS = [
    ("SPY", "stock"),
    ("I:SPX", "index"),
    ("I:VIX", "index"),
]

display(
    pd.DataFrame(
        [
            {"year": year, "probe_date": date}
            for year, dates in PROBE_DATES.items()
            for date in dates
        ]
    )
)

## 3. Run the non-destructive API probe

In [ ]:
def probe_one_day(ticker: str, asset_class: str, probe_date: str) -> dict:
    result = {
        "ticker": ticker,
        "asset_class": asset_class,
        "probe_date": probe_date,
        "year": int(probe_date[:4]),
        "status": None,
        "raw_rows": 0,
        "normalised_rows": 0,
        "first_timestamp": None,
        "last_timestamp": None,
        "error": None,
    }

    try:
        raw = client.aggregates(
            ticker,
            probe_date,
            probe_date,
        )

        raw_rows = len(raw) if raw is not None else 0
        result["raw_rows"] = raw_rows

        if not raw:
            result["status"] = "NO_ROWS"
            return result

        frame = normalize_aggregates(
            raw,
            ticker,
            asset_class,
        )

        result["normalised_rows"] = len(frame)

        if len(frame):
            timestamp_col = (
                "timestamp_et"
                if "timestamp_et" in frame.columns
                else (
                    "timestamp_utc"
                    if "timestamp_utc" in frame.columns
                    else None
                )
            )

            if timestamp_col:
                result["first_timestamp"] = frame[timestamp_col].min()
                result["last_timestamp"] = frame[timestamp_col].max()

        result["status"] = "AVAILABLE"
        return result

    except Exception as exc:
        result["status"] = "ERROR_OR_ENTITLEMENT"
        result["error"] = str(exc)[:1000]
        return result


rows = []

for ticker, asset_class in TICKERS:
    for year, dates in PROBE_DATES.items():
        for probe_date in dates:
            print(f"Testing {ticker} on {probe_date}...")
            rows.append(
                probe_one_day(
                    ticker=ticker,
                    asset_class=asset_class,
                    probe_date=probe_date,
                )
            )

probe_results = pd.DataFrame(rows)

display(probe_results)

## 4. Summarise access by ticker and year

In [ ]:
summary_rows = []

for (ticker, asset_class, year), group in probe_results.groupby(
    ["ticker", "asset_class", "year"]
):
    statuses = set(group["status"].dropna())

    if statuses == {"AVAILABLE"}:
        overall = "AVAILABLE"
    elif "AVAILABLE" in statuses:
        overall = "MIXED"
    elif statuses == {"NO_ROWS"}:
        overall = "NO_ROWS"
    elif statuses == {"ERROR_OR_ENTITLEMENT"}:
        overall = "ERROR_OR_ENTITLEMENT"
    else:
        overall = "MIXED"

    summary_rows.append(
        {
            "ticker": ticker,
            "asset_class": asset_class,
            "year": year,
            "overall_status": overall,
            "probe_dates": len(group),
            "available_dates": int(group["status"].eq("AVAILABLE").sum()),
            "no_row_dates": int(group["status"].eq("NO_ROWS").sum()),
            "error_dates": int(
                group["status"].eq("ERROR_OR_ENTITLEMENT").sum()
            ),
            "max_rows_returned": int(group["normalised_rows"].max()),
            "example_error": (
                group.loc[
                    group["error"].notna(),
                    "error",
                ].iloc[0]
                if group["error"].notna().any()
                else None
            ),
        }
    )

access_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(["year", "ticker"], ascending=[False, True])
    .reset_index(drop=True)
)

display(access_summary)

## 5. Simple decision table

In [ ]:
decision = access_summary.pivot(
    index="ticker",
    columns="year",
    values="overall_status",
)

display(decision)

for year in [2022, 2021]:
    year_rows = access_summary[
        access_summary["year"].eq(year)
    ]

    all_available = (
        len(year_rows) == len(TICKERS)
        and year_rows["overall_status"].isin(
            ["AVAILABLE", "MIXED"]
        ).all()
    )

    if all_available:
        print(
            f"{year}: all three underlying series returned data on at least "
            "one probe date. A fuller historical retrieval is worth testing."
        )
    else:
        print(
            f"{year}: at least one required underlying did not return usable "
            "data in this probe. Review the per-ticker statuses/errors above "
            "before extending the dissertation pipeline."
        )

## 6. Save the probe evidence

These files are useful for documenting exactly what the API returned on the day the historical-access check was performed.

In [ ]:
detail_path = OUTPUT_ROOT / "massive_2021_2022_probe_detail.csv"
summary_path = OUTPUT_ROOT / "massive_2021_2022_probe_summary.csv"
manifest_path = OUTPUT_ROOT / "massive_2021_2022_probe_manifest.json"

probe_results.to_csv(detail_path, index=False)
access_summary.to_csv(summary_path, index=False)

manifest = {
    "purpose": "Check historical SPY/SPX/VIX minute-data access before expanding dissertation history",
    "years_tested": sorted(PROBE_DATES),
    "tickers": [ticker for ticker, _ in TICKERS],
    "probe_dates": PROBE_DATES,
    "database_modified": False,
    "raw_data_folders_modified": False,
    "interpretation_note": (
        "The probe reports observed API responses. A successful date indicates "
        "that historical aggregate data were returned for that ticker/date; "
        "it does not itself guarantee every trading day in the year is complete."
    ),
}

manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("Saved:")
print(" -", detail_path)
print(" -", summary_path)
print(" -", manifest_path)

## 7. What to do with the result

### If 2022 and 2021 both work for SPY, SPX and VIX

Do **not** immediately mix them into the already-completed RQ5 options backtest. Instead:

1. change the extended underlying start date;
2. rebuild RQ1–RQ4 with the same feature definitions and temporal methodology;
3. describe the longer history as an underlying-market robustness extension;
4. leave the two-year options/RQ5 period unchanged because option availability is a separate constraint.

### If SPY works but one or both index series fail

The current dissertation feature set cannot be recreated consistently for that year, so do not silently substitute a different source in the same robustness analysis.

### If the API returns an entitlement error

Keep the exact error in the saved probe report. It documents that the limitation came from available historical access rather than an arbitrary analytical choice.